In [30]:
import urllib.parse
import requests
from bs4 import BeautifulSoup
from clean_url import sanitize_and_generate_dynamic_folder

In [31]:
def get_urls_from_sitemap(start_url: str) -> set:
    urls = set()

    # Determine base root (e.g. "https://www.digisinc.systems")
    parsed = urllib.parse.urlparse(start_url)
    base_url = f"{parsed.scheme}://{parsed.netloc}"

    # Standard sitemap paths to check
    possible_sitemaps = [
        f"{base_url}/sitemap.xml",
        f"{base_url}/sitemap_index.xml",
    ]

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }

    for sitemap_url in possible_sitemaps:
        print(f"Checking: {sitemap_url}")
        try:
            res = requests.get(sitemap_url, headers=headers, timeout=10)
            if res.status_code == 200:
                # Parse XML response
                soup = BeautifulSoup(res.content, "xml")

                # Look for standard <loc> tags containing the page URLs
                locations = soup.find_all("loc")
                for loc in locations:
                    url = loc.text.strip()
                    urls.add(url)

                if urls:
                    print(
                        f" Successfully found {len(urls)} URLs in {sitemap_url}"
                    )
                    break

        except requests.RequestException as e:
            print(f"Could not fetch {sitemap_url}: {e}")

    return urls

In [32]:
raw_start_url = "https:/www.digisinc.systems"

# 1. Clean raw URL
sanitized_url, _ = sanitize_and_generate_dynamic_folder(raw_start_url)

# 2. Extract URLs from Sitemap
if sanitized_url:
    discovered_urls = get_urls_from_sitemap(sanitized_url)

    if discovered_urls:
        print("\nDiscovered Pages from Sitemap:")
        for url in sorted(discovered_urls):
            print(url)
    else:
        print(
            "\nNo sitemap found or sitemap was empty. Fall back to Playwright or crawler."
        )

Checking: https://www.digisinc.systems/sitemap.xml
 Successfully found 5 URLs in https://www.digisinc.systems/sitemap.xml

Discovered Pages from Sitemap:
https://digisinc.systems/
https://digisinc.systems/chat
https://digisinc.systems/contact
https://digisinc.systems/projects
https://digisinc.systems/services
